# IBKR API notebook

#### Connection

In [1]:
from ib_async import *
import pandas as pd
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)

<IB connected to 127.0.0.1:7497 clientId=14>

Error 162, reqId 4: Message d'erreur Service Donn\u00e9es de March\u00e9 Historiques:La demande HMDS n'a fourni aucune donn\u00e9e\u00a0: NDX@NASDAQ Trades, contract: Index(symbol='NDX', exchange='NASDAQ', currency='USD')
Error 162, reqId 5: Message d'erreur Service Donn\u00e9es de March\u00e9 Historiques:La demande HMDS n'a fourni aucune donn\u00e9e\u00a0: NDX@NASDAQ Trades, contract: Index(symbol='NDX', exchange='NASDAQ', currency='USD')


## Request Historical data

#### Choose your contract

In [ ]:
contract = Stock('AAPL', 'SMART', 'USD')

In [2]:
contract = Index('NDX', 'NASDAQ', 'USD')

In [ ]:
contract = Forex('EURUSD')

#### Check first data timestamp available

In [3]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=True)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
print(f"First date of data available: {formatted_time}")

First date of data available: March 04, 2004, 14:30


#### Request historical data function

In [4]:
save_path = "../database/NDX_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)
# Retrieve the first date from your existing DataFrame
first_date = retrieved_df.iloc[0]['date']  # Assuming 'date' is the column name
end_date = first_date.strftime('%Y%m%d %H:%M:%S')  # Format as 'yyyyMMdd HH:mm:ss'
print(f"First date in the DataFrame: {first_date}")
print(f"End date: {end_date}")

First date in the DataFrame: 2022-01-31 09:30:10-05:00
End date: 20220131 09:30:10


In [ ]:
#today's date
end_date = pd.Timestamp.now().strftime('%Y%m%d %H:%M:%S')

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [7]:
historical_data_interval = '10 secs' 
request_duration = '1 M'
bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1,
        timeout = 0)

In [6]:
bars[0]

IndexError: list index out of range

Convert the list of bars to a data frame and print the first and last rows:

In [ ]:
df = util.df(bars)

display(df.head(n=20))
display(df.tail(n=20))

Save your pulled data in a dataframe

Compression possibilities sorted by compression ratio from the lowest to the highest : 
- `snappy`

- `gzip`

- `brotli`

#### Checking if volume and average columns are empty or not and remove it if empty

In [ ]:
# Check if the 'volume' column is empty (all values are 0.0)
if (df['volume'] == 0.0).all():
    df = df.drop(columns=['volume'])  # Drop the 'volume' column
    print("The 'volume' column was empty and has been removed.")

# Check if the 'average' column is empty (all values are 0.0)
if (df['average'] == 0.0).all():
    df = df.drop(columns=['average'])  # Drop the 'average' column
    print("The 'average' column was empty and has been removed.")

# Display the updated DataFrame
display(df.head(n=30))

Construction du nom du fichier et sauvegarde en `.parquet` dans le dossier `database`

Structure du nom du fichier : `Symbol_Interval_StartDate_EndDate.parquet`

In [ ]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol

# Construire le nom du fichier
start_date = df['date'].iloc[0].strftime('%Y%m%d')
end_date = df['date'].iloc[-1].strftime('%Y%m%d')
# Structure
save_path = f"../database/{symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}.parquet"

# Sauvegarder le DataFrame en fichier parquet
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

In [9]:
save_path = "../database/NDX_1min_20050411_to_20250404.parquet"
#save_path = "../database/NDX_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)

# Display the first and last rows of the DataFrame
display(retrieved_df.head(10))
display(retrieved_df.tail(10))

,date,open,high,low,close,barCount
0,2005-04-11 09:30:00-04:00,1489.88,1491.31,1489.30,1491.31,88
1,2005-04-11 09:31:00-04:00,1491.42,1492.15,1491.42,1492.00,23
2,2005-04-11 09:32:00-04:00,1492.12,1492.27,1491.37,1491.37,21
3,2005-04-11 09:33:00-04:00,1491.25,1491.63,1490.99,1491.25,12
4,2005-04-11 09:34:00-04:00,1491.38,1492.22,1491.26,1492.22,11
5,2005-04-11 09:35:00-04:00,1492.33,1492.45,1492.11,1492.20,11
6,2005-04-11 09:36:00-04:00,1492.32,1492.32,1491.36,1491.36,10
7,2005-04-11 09:37:00-04:00,1491.47,1491.84,1491.36,1491.69,12
8,2005-04-11 09:38:00-04:00,1491.80,1491.80,1490.91,1490.91,19
9,2005-04-11 09:39:00-04:00,1490.80,1490.80,1489.83,1489.83,16


,date,open,high,low,close,barCount
1953877,2025-04-04 10:16:00-04:00,17821.28,17829.97,17801.41,17826.91,60
1953878,2025-04-04 10:17:00-04:00,17830.03,17860.95,17824.10,17848.94,60
1953879,2025-04-04 10:18:00-04:00,17848.03,17855.07,17828.33,17846.50,60
1953880,2025-04-04 10:19:00-04:00,17845.04,17884.30,17841.19,17874.68,60
1953881,2025-04-04 10:20:00-04:00,17872.75,17904.64,17865.03,17891.17,60
1953882,2025-04-04 10:21:00-04:00,17890.45,17916.59,17883.70,17906.47,60
1953883,2025-04-04 10:22:00-04:00,17905.35,17905.36,17880.04,17884.72,60
1953884,2025-04-04 10:23:00-04:00,17882.37,17882.38,17847.59,17848.76,60
1953885,2025-04-04 10:24:00-04:00,17842.78,17851.01,17817.60,17828.49,60
1953886,2025-04-04 10:25:00-04:00,17828.25,17845.12,17816.55,17837.65,60


Instruct the notebook to draw plot graphics inline:

In [ ]:
%matplotlib inline

Plot the close data

In [ ]:
df.plot(y='close');

There is also a utility function to plot bars as a candlestick plot. It can accept either a DataFrame or a list of bars. Here it will print the last 100 bars:

In [ ]:
util.barplot(bars[-100:], title=contract.symbol);

## Historical data with realtime updates

A new feature of the API is to get live updates for historical bars. This is done by setting `endDateTime` to an empty string and the `keepUpToDate` parameter to `True`.

Let's get some bars with an keepUpToDate subscription:

In [ ]:

bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='900 S',
        barSizeSetting='10 secs',
        whatToShow='MIDPOINT',
        useRTH=True,
        formatDate=1,
        keepUpToDate=True)

Replot for every change of the last bar:

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def onBarUpdate(bars, hasNewBar):
    plt.close()
    plot = util.barplot(bars)
    clear_output(wait=True)
    display(plot)

bars.updateEvent += onBarUpdate

ib.sleep(10)
ib.cancelHistoricalData(bars)

Realtime bars
------------------

With ``reqRealTimeBars`` a subscription is started that sends a new bar every 5 seconds.

First we'll set up a event handler for bar updates:

In [ ]:
def onBarUpdate(bars, hasNewBar):
    print(bars[-1])

Then do the real request and connect the event handler,

In [ ]:
bars = ib.reqRealTimeBars(contract, 5, 'MIDPOINT', False)
bars.updateEvent += onBarUpdate

let it run for half a minute and then cancel the realtime bars.

In [ ]:
ib.sleep(30)
ib.cancelRealTimeBars(bars)

The advantage of reqRealTimeBars is that it behaves more robust when the connection to the IB server farms is interrupted. After the connection is restored, the bars from during the network outage will be backfilled and the live bars will resume.

reqHistoricalData + keepUpToDate will, at the moment of writing, leave the whole API inoperable after a network interruption.

### Request historical market news

In [ ]:
ib.reqHistoricalNews()
#ib.reqHistoricalNewsAsync()

In [ ]:
ib.disconnect()